# Large Reasoning Model Fine-Tuning for Silkome Reasoning

By: Jay Siri (jaysiri@mit.edu)

This notebook finetunes open-source LLMs (Qwen-Instruct) using parameter-efficient fine-tuning to learn to reason about silk fiber mechanical properties.

**Fine-Tuning Procedure:**

1. SFT on GB1 (sequences → fitness scores, AA count, motif counts)
2. SFT on ProteinGym (sequences → fitness scores, AA count, motif counts)
3. GRPO on GB1
4. CLM on spindroin literature
5. SFT(/ORPO) on GPT-5.2 responses of Silkome train
6. SFT on Silkome train (sequences → strength/toughness scores, AA counts, motif counts)
7. GRPO on Silkome train


### **0. Setup**

Install necessary libraries, define the system prompts, and load in the base model.

In [ ]:
!pip install -U transformers datasets peft trl accelerate
!pip install wandb
!pip install --upgrade torchao

In [ ]:
import wandb
import os, re, json, random
import requests
import torch
import pyarrow as pa
import io

from typing import Optional, Tuple, Dict, List, Any
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, AutoPeftModelForCausalLM
from trl import SFTTrainer, SFTConfig, GRPOTrainer, GRPOConfig
from google.colab import drive

SEED = int(os.environ.get("SEED", "42"))
set_seed(SEED)
random.seed(SEED)

# -------------------------------
# Paths/Model
# -------------------------------
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
OUT_ROOT   = "outputs"
SFT_GB1_DIR    = f"{OUT_ROOT}/{BASE_MODEL}-gb1-sft"
SFT_PROTEINGYM_DIR    = f"{OUT_ROOT}/{BASE_MODEL}-protein-gym-sft"
GRPO_GB1_DIR   = f"{OUT_ROOT}/{BASE_MODEL}-gb1-grpo"
CLM_SPIDROIN_DIR  = f"{OUT_ROOT}/{BASE_MODEL}-spindroin-clm"
SFT_GPT5_DIR  = f"{OUT_ROOT}/{BASE_MODEL}-gpt5-sft"
SFT_SILK_DIR  = f"{OUT_ROOT}/{BASE_MODEL}-silk-sft"
GRPO_SILK_DIR  = f"{OUT_ROOT}/{BASE_MODEL}-silk-grpo"

# -------------------------------
# XML schemas
# -------------------------------
SYSTEM_PROMPT_GB1 = """Respond in this exact XML format:
<reasoning>
Briefly analyze how sequence features (e.g., hydrophobic/charged balance, proline/glycine content) affect binding fitness.
</reasoning>
<answer>
{"fitness": <float 0..1>}
</answer>
"""

SYSTEM_PROMPT_SILK = """Respond in this exact XML format:
<reasoning>
Explain how sequence features impact strength and toughness.
</reasoning>
<answer>
{"strength": <float in the range [0, 1]>, "toughness": <float in the range [0, 1]>}
</answer>
"""


# -------------------------------
# Example Input
# -------------------------------
EXAMPLE_INPUT = """Your task is to consider a protein sequence in FASTA format, along with the family, genus, species, and protein category to predict the strength and toughness of silk fibers produced by the spider.
Please analyze the following data:

Protein sequence: SSSAYSGTSTGGSSVSQSQPIISSAPVYFNAQTLTSSLASSLQSDRALNFISSGQLSASDVSTSVSSAVAQTLGISQSSVQNIISQQMSSVRTGASSSSVSQAIANAVSSAVQASGAATPGQEQSIAQRVYSSISTYLSQLISQRTAPAPAPAPRPAPMPAPAPRPAPMPAPAPRPRPAPAPRPAPVYAPAPVVSQIQAAASSQSSAQQSSFAQAQQSAYAQSQQSSSAYSGASTGGSSVSQSQPIISSAPVYFNAQTLTSSLASSLQSDRALNFISSGQLSASDVSTSVSSAVAQTLGISQSSVQNIISQQMSSVRTGASSSSVSQAIANAVSSAVQASGAATPGQEQSIAQRVYSSISTYLSQLISQRTAPAPAPAPRPAPMPAPAPRPAPMPAPAPRPRPAPAPRPASVYAPAPVVSQIQAAASSQSSSQQSSFAQAQQSAYAQSQQSSSAYSGASTGGSSVSQSQPIISSAPVYFNAQTLTSSLASSLQSDRALNFISSGQLSASDVSTSVSSAVAQTLGISQSSVQNIISQQMSSVRTGASSSSVSQAIANAVSSAVQASGAATPGQEQSIAQRVYSSISTYLSQLISQRTAPAPAPAPRPAPMPAPAPRPAPMPAPAPRPRPAPAPRPAPVYAPAPVVSQIQAAASSQSSAQQSSFAQAQQSAYAQSQQSSSAYSGASTGGSSVSQSQPIISSAPVYFNAQTLTSSLASSLQSDRALNFISSGQLSASDVSTSVSSAVAQTLGISQSSVQNIISQQMSSVRTGASSSSVSQAIANAVSSAVQASGAATPGQEQSIAQRVYSSISTYLSQLISQRTAPAPAPAPRPAPMPAPAPRPAPMPAPAPRPRPAPAPRPAPVYAPAPVVSQIQAAASSQSSSQQSSFAQAQQSAYAQSQQSSSAYSGASTGGSSVSQSQPIASSAPVYFNTQTLSSSLASSLQSDRALNFISSGQLSAADVSTGVSSAVAQALGISQSSVQNIISQQMSSIRTGASSSSVSQAIANAVSSAVQASGAAAPGQEQGIAQSVYSAISTYLSQLISQRTAPAPAPAPRPLPAPMPAPAPRPRPAPAPRPAPIYAPAPVVSQLQTASSSQSSAQQNSFAQSQQSAFAQSQQSSIAAQSQQRQSANAYSTAPSFAGSQVQQAAAAQSQVSASSFSSGSSPGIGSSPSSSAAFSAPISSAPYAPSVVASSSSSVGPSSGSALAASAAQQLTSAAANQRIAALSNSLRSAVAGGQVNYGALSNSLASAASQIQRSSGLSKSEVLVEALLETLAALLDSLSTSGSSSSQFAQAVLQAFA
Family: Araneidae
Genus: Eriophora
Species: pustulosa
Protein category: PySp.

Now predict the strength and toughness of the silk fibers, each in a range between 0.0 and 1.0."""

In [ ]:
device = torch.device("cuda")
print(torch.__version__)
print(torch.cuda.is_available(), torch.cuda.device_count())

In [ ]:
def generate_example_prediction():
  """
  Utility function used to generate and examine a single example prediction on the current model.
  """
  print("Generating example prediction ...")
  generated = torch.tensor(tokenizer.encode(EXAMPLE_INPUT + "\n" + SYSTEM_PROMPT_SILK)).unsqueeze(0).to(device)
  sample_outputs = model.generate(
                    inputs=generated,
                    max_new_tokens=1500,
                    min_new_tokens=20,
                    pad_token_id=tokenizer.pad_token_id,
                    temperature=0.5,
                    use_cache=False
                    )
  decoded_text = tokenizer.decode(sample_outputs[-1], skip_special_tokens=True)

  print(decoded_text)


In [ ]:
# -------------------------------
# Tokenizer + Base + LoRA
# -------------------------------
print("Loading base model + LoRA ...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.padding_side='left'

# Add custom tokenization of important vocabulary/motifs
tokenizer.add_tokens(['GPGG', 'GPGGX', 'GPGGA', 'GPGGL', 'GPGGQ', 'GPGGY', 'GG', 'GGX', 'GGA', 'GGL', 'GGQ', 'GGY', 'AAAAAA', 'QQ', 'tubuliform', 'flagelliform', 'ampullate', 'MiSp', 'MaSp', 'aciniform', 'AcSP', 'pyriform', 'PySp', 'AgSp'])

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token
    tokenizer.pad_token_id = tokenizer.unk_token_id

# Load model
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")

# Load LoRA adapters
lora_cfg = LoraConfig(task_type="CAUSAL_LM", r=16, lora_alpha=16, target_modules="all-linear", lora_dropout=0.05)
model = get_peft_model(model, lora_cfg)
model.resize_token_embeddings(len(tokenizer))

model.print_trainable_parameters()

In [ ]:
# Verify everything is working...
generate_example_prediction()

### **1. SFT on GB1**

SFT (https://huggingface.co/docs/trl/en/sft_trainer) on SaProtHub/Dataset-GB1-fitness, which maps GB1 sequences to fitness scores.

The "input" is the system prompt + the user input (in this case, the GB1 protein sequnce), and the "output" is the fitness score along with synthetic hints defined below.

In [ ]:
# -------------------------------
# Load GB1 (simple HF dataset) and normalize target to [0,1]
# -------------------------------
print("Loading SaProtHub/Dataset-GB1-fitness …")
gb1 = load_dataset("SaProtHub/Dataset-GB1-fitness")
gb1_train = gb1["train"]
gb1_valid = gb1["valid"] if "valid" in gb1 else (gb1.get("validation") or None)

if MAX_GB1_SAMPLES and len(gb1_train) > MAX_GB1_SAMPLES:
    gb1_train = gb1_train.select(range(MAX_GB1_SAMPLES))

# Infer column names robustly
cols = gb1_train.column_names
SEQ_COL   = "sequence" if "sequence" in cols else cols[0]
LABEL_COL = "label"    if "label"    in cols else ("fitness" if "fitness" in cols else cols[1])

In [ ]:
# Min–max on TRAIN only → map to [0,1] as target for JSON
train_vals = [float(x[LABEL_COL]) for x in gb1_train]
mn, mx = min(train_vals), max(train_vals)

def _norm(v):
    return 0.5 if mx == mn else max(0.0, min(1.0, (float(v)-mn)/(mx-mn)))

def add_norm(ex):
    ex["fitness01"] = _norm(ex[LABEL_COL]);
    return ex

gb1_train = gb1_train.map(add_norm)
gb1_valid = gb1_valid.map(add_norm) if gb1_valid else None

In [ ]:
# -------------------------------
# Tiny synthetic rationale for SFT
# -------------------------------
def aa_stats(seq: str) -> Dict[str, float]:
    s = (seq or "").upper().replace(" ", "")
    L = max(1, len(s))
    def frac(chars): return sum(c in chars for c in s)/L
    return {
        "hydrophobic": frac(set("AILMVFWY")),
        "polar":       frac(set("STNQCY")),
        "positive":    frac(set("KRH")),
        "negative":    frac(set("DE")),
        "proline":     s.count("P")/L,
        "glycine":     s.count("G")/L,
    }

def synth_reasoning(seq: str, f01: float) -> str:
    st = aa_stats(seq)
    trend = "higher" if f01 >= 0.6 else ("lower" if f01 <= 0.4 else "moderate")
    hints = []
    if st["hydrophobic"] > 0.45: hints.append("hydrophobic core packing likely improves stability")
    if st["proline"] > 0.08:     hints.append("proline can disrupt helices but stabilize turns")
    if st["glycine"] > 0.15:     hints.append("glycine increases flexibility; excessive amounts may reduce stability")
    if not hints: hints.append("balanced charged/polar residues support solubility and binding accessibility")
    return (f"Amino-acid composition suggests {trend} fitness. "
            f"hydrophobic={st['hydrophobic']:.2f}, polar={st['polar']:.2f}, "
            f"positive={st['positive']:.2f}, negative={st['negative']:.2f}, "
            f"P={st['proline']:.2f}, G={st['glycine']:.2f}. " + "; ".join(hints) + ".")

In [ ]:
# -------------------------------
# Helper: chat template
# -------------------------------
def apply_chat_template(tokenizer, system: str, user: str, assistant: Optional[str] = None) -> str:
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": user}]
    if assistant is not None:
        messages.append({"role": "assistant", "content": assistant})
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

In [ ]:
# -------------------------------
# Build SFT texts (teacher-forced XML)
# -------------------------------
def gb1_sft_row(ex):
    seq = ex[SEQ_COL]
    f01 = ex["fitness01"]
    user = f"Given the GB1 mutant sequence below, predict normalized fitness in [0,1] with a mechanistic reason.\nSequence:\n{seq}"
    assistant_xml = f"<reasoning>{synth_reasoning(seq, f01)}</reasoning><answer>{{\"fitness\": {f01:.6f}}}</answer>"
    return {"text": apply_chat_template(tokenizer, SYSTEM_PROMPT_GB1, user, assistant_xml)}

gb1_sft = gb1_train.map(gb1_sft_row, remove_columns=[c for c in gb1_train.column_names if c != "text"])

In [ ]:
# -------------------------------------------
# SFT on GB1
# -------------------------------------------
print("Running SFT on GB1 …")
sft_args = SFTConfig(
    output_dir=SFT_GB1_DIR,
    run_name="gb1_sft",
    learning_rate=1e-5,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=10,
    gradient_accumulation_steps=8,
    bf16=True,
    logging_steps=10,
    save_steps=10,
    num_train_epochs=1,
    dataset_text_field="text",
    packing=True,
    eos_token="<|im_end|>"
)
sft_tr = SFTTrainer(model=model, args=sft_args, train_dataset=gb1_sft)
sft_tr.train()

In [ ]:
generate_example_prediction()

### **2. SFT on ProteinGym_v1**

Similar to above, but on a larger dataset (OATML-Markslab/ProteinGym_v1).

In [ ]:
# -------------------------------
# Load GB1 (simple HF dataset) and normalize target to [0,1]
# -------------------------------
print("Loading OATML-Markslab/ProteinGym_v1 …")
proteingym1 = load_dataset("OATML-Markslab/ProteinGym_v1", data_dir='DMS_substitutions')
proteingym_train = proteingym1["train"]
proteingym_valid = proteingym1["valid"] if "valid" in proteingym1 else (proteingym1.get("validation") or None)
cols = proteingym_train.column_names
SEQ_COL   = "mutant" if "mutant" in cols else cols[0]
LABEL_COL = "DMS_score"    if "DMS_score"    in cols else ("DMS_score" if "DMS_score" in cols else cols[1])

In [ ]:
# -------------------------------
# Build SFT texts (teacher-forced XML)
# -------------------------------
def proteingym_sft_row(ex):
    seq = ex[SEQ_COL]
    f01 = ex["fitness01"]
    user = f"Given the protein mutant sequence below, predict normalized fitness in [0,1] with a mechanistic reason.\nSequence:\n{seq}"
    assistant_xml = f"<reasoning>{synth_reasoning(seq, f01)}</reasoning><answer>{{\"fitness\": {f01:.6f}}}</answer>"
    return {"text": apply_chat_template(tokenizer, SYSTEM_PROMPT_GB1, user, assistant_xml)}

# Min–max on TRAIN only → map to [0,1] as target for JSON
train_vals = [float(x[LABEL_COL]) for x in proteingym_train]
mn, mx = min(train_vals), max(train_vals)
proteingym_train = proteingym_train.map(add_norm)
proteingym_valid = proteingym_valid.map(add_norm) if proteingym_valid else None
proteingym_sft = proteingym_train.map(proteingym_sft_row, remove_columns=[c for c in proteingym_train.column_names if c != "text"])

In [ ]:
# -------------------------------------------
# SFT on ProteinGym
# -------------------------------------------
print("SFT on ProteinGym …")
sft_args = SFTConfig(
    output_dir=SFT_PROTEINGYM_DIR,
    run_name="proteingym_sft",
    learning_rate=1e-5,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=10,
    gradient_accumulation_steps=8,
    bf16=True,
    logging_steps=10,
    save_steps=10,
    num_train_epochs=1,
    dataset_text_field="text",
    packing=True,
    report_to="wandb",
    eos_token="<|im_end|>"
)
sft_tr = SFTTrainer(model=model, args=sft_args, train_dataset=proteingym_sft)
sft_tr.train()

In [ ]:
generate_example_prediction()

### **3. GRPO on GB1**

GRPO (https://huggingface.co/docs/trl/en/grpo_trainer) is used to try to encourage the model to reason.

We define verifiable rewards (below the rewards are defined for: answer format, answer accuracy, are reasoning response length). The model then generates responses for each prompt (`num_generations` in the GRPO Trainer Config) and then learns from the rewards using the GRPO algorithm.

In [ ]:
# ============================================================
# Rewards for GRPO
# ============================================================
XML_RE = re.compile(r"^\s*<reasoning>\s*(?P<r>[\s\S]*?)\s*</reasoning>\s*<answer>\s*(?P<a>[\s\S]*?)\s*</answer>\s*$", re.S)

def _extract_sections(text: str) -> Tuple[Optional[str], Optional[str]]:
    m = XML_RE.match(text.strip())
    return (m.group("r"), m.group("a")) if m else (None, None)

def parse_json_from_completions(completions: List[List[Dict]]) -> List[Optional[Dict]]:
    outs = []
    for c in completions:
        t = c[0]["content"]
        _, ans = _extract_sections(t)
        if ans is None:
            outs.append(None); continue
        m = re.findall(r"\{[\s\S]*\}", ans)
        if not m:
            outs.append(None); continue
        jtxt = m[-1]
        try:
            outs.append(json.loads(jtxt))
        except Exception:
            try:
                jtxt2 = re.sub(r"[`’“”]", '"', jtxt)
                jtxt2 = re.sub(r"%", "", jtxt2)
                outs.append(json.loads(jtxt2))
            except Exception:
                outs.append(None)
    return outs

# Robust number coercion (handles lists/dicts/strings) and clamps to [0,1]
FLOAT_RE = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")
def coerce_scalar01(x):
    if isinstance(x, (list, tuple)):
        for cand in list(x)[::-1]:
            v = coerce_scalar01(cand)
            if v is not None: return v
        return None
    if isinstance(x, dict):
        for k in ("value","val","mean","median","fitness","score","strength","toughness"):
            if k in x:
                return coerce_scalar01(x[k])
        return None
    if isinstance(x, str):
        m = FLOAT_RE.search(x)
        if not m: return None
        x = m.group(0)
    try:
        v = float(x)
    except Exception:
        return None
    return max(0.0, min(1.0, v))

def anti_shortcut_penalty(completions, **kwargs):
    pen = []
    for c in completions:
        t = c[0]["content"]; r, _ = _extract_sections(t)
        bad = (r is None) or (len(re.findall(r"\w+|\S", r)) < 30)
        pen.append(-0.2 if bad else 0.0)
    wandb.log({"anti_shortcut_penalty_pen": pen})
    return pen

def json_field_reward_gb1(completions, **kwargs):
    outs = parse_json_from_completions(completions)
    scores = []
    for o in outs:
        v = coerce_scalar01(o.get("fitness")) if isinstance(o, dict) else None
        scores.append(0.05 if v is not None else 0.0)
    wandb.log({"json_field_reward_gb1_scores": scores})
    return scores

# ---- completion→gold mapping helpers ----
def _flatten_ids(x: Any) -> List[int]:
    out = []
    if isinstance(x, (list, tuple)):
        for e in x:
            out.extend(_flatten_ids(e))
    elif x is not None:
        try: out.append(int(x))
        except Exception: pass
    return out

def _gold_index_for_i(i: int, B: int, N: int, completion_ids_flat: Optional[List[int]]) -> int:
    # Use provided IDs if they look like local [0..B-1]; else fall back to shape-based mapping.
    if completion_ids_flat and i < len(completion_ids_flat):
        idx = completion_ids_flat[i]
        if 0 <= idx < B:
            return idx
    # Fallback: assume completions are grouped by prompt: G completions per prompt
    G = max(1, N // max(1, B))
    idx = i // G if G > 0 else (i % max(1, B))
    if idx >= B: idx = idx % B
    return idx

def gated_closeness_gb1(prompts, completions, answer, completion_ids=None, **kwargs):
    outs = parse_json_from_completions(completions)
    N = len(outs)
    B = len(answer)
    flat_ids = _flatten_ids(completion_ids)
    scores = []
    for i in range(N):
        t = completions[i][0]["content"]
        r, _ = _extract_sections(t)
        if (r is None) or (len(re.findall(r"\w+|\S", r)) < 60):
            scores.append(0.0); continue
        gold_idx = _gold_index_for_i(i, B, N, flat_ids)
        gold = answer[gold_idx] if 0 <= gold_idx < B else None
        pred = outs[i]
        p = coerce_scalar01(pred.get("fitness")) if isinstance(pred, dict) else None
        g = coerce_scalar01(gold.get("fitness")) if isinstance(gold, dict) else None
        if p is None or g is None:
            scores.append(0.0); continue
        scores.append(max(0.0, 1.0 - abs(p - g)))
    wandb.log({"gated_closeness_gb1_scores": scores})
    return scores

In [ ]:
# -------------------------------
# Build GRPO texts
# -------------------------------
def gb1_grpo_row(ex):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT_GB1},
            {"role": "user", "content": f"Predict normalized GB1 fitness in [0,1] with reasoning.\nSequence:\n{ex[SEQ_COL]}"},
        ],
        "answer": {"fitness": float(ex["fitness01"])},
    }
gb1_grpo = gb1_train.map(gb1_grpo_row, remove_columns=[c for c in gb1_train.column_names if c not in []])

In [ ]:
# ============================================================
# GRPO on GB1
# ============================================================
print("Running GRPO on GB1 ...")
grpo_args_gb1 = GRPOConfig(
    output_dir=GRPO_GB1_DIR,
    run_name="gb1_grpo",
    learning_rate=5e-6,
    adam_beta1=0.9, adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    bf16=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=8,
    num_generations=4,
    max_completion_length=384,
    num_train_epochs=1,
    save_steps=10,
    max_grad_norm=0.1,
    temperature = 0.5,
)

gb1_tr = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[json_field_reward_gb1, anti_shortcut_penalty, gated_closeness_gb1],
    args=grpo_args_gb1,
    train_dataset=gb1_grpo,
)
gb1_tr.train()

In [ ]:
generate_example_prediction()

### **4. CLM on Spindroin Literature**

Causal Language Model (CLM) trains the model to do next-token prediction so it can learn from a corpus of data.



In [ ]:
SPIDROIN_TXT_URL = 'https://raw.githubusercontent.com/jaypsiri/Silkome-Reasoning/refs/heads/main/data/cleaned_spindroin_literature.txt'
spidroin_texts_response = requests.get(SPIDROIN_TXT_URL)
spidroin_text_content = spidroin_texts_response.text
literature_dataset = Dataset.from_list([{"text": spidroin_text_content}])

In [ ]:
def tokenize(example):
    return tokenizer(example["text"])

tokenized_dataset = literature_dataset.map(tokenize, batched=True, remove_columns=["text"])

In [ ]:
block_size = 512

def group_texts(examples):
    """Concatenate all texts into chunked sizes to process."""
    concatenated = sum(examples["input_ids"], [])
    total_length = (len(concatenated) // block_size) * block_size

    result = {
        "input_ids": [
            concatenated[i : i + block_size]
            for i in range(0, total_length, block_size)
        ]
    }

    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_dataset.map(group_texts, batched=True, remove_columns=["attention_mask"])

In [ ]:
clm_training_args = TrainingArguments(
    output_dir=CLM_SPIDROIN_DIR,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=1e-5,
    weight_decay=0.01,
    logging_steps=2,
    save_steps=2,
)
print("Running CLM on spindroin literature texts...")
clm_trainer = Trainer(
    model=model,
    args=clm_training_args,
    train_dataset=lm_dataset,
)
clm_trainer.train()

In [ ]:
generate_example_prediction()

### **5. SFT/ORPO on Silkome Reasoning**

Use a smaller dataset of ~800 GPT-5.2 responses to Silkome training to finetune the model.

In [ ]:
GPT_SILKOME_URL = 'https://raw.githubusercontent.com/jaypsiri/Silkome-Reasoning/refs/heads/main/data/gpt52_silkome.json'
gpt_silkome_response = requests.get(GPT_SILKOME_URL)
gpt_silkome_content = gpt_silkome_response.json()

gpt_sft = []

for item in gpt_silkome_content:
    system_prompt = item.get("input", dict().get("content")).get("prompt", None)[0].get('content')
    user_prompt = item.get("input", dict().get("content")).get("prompt", None)[1].get('content')
    chosen_reasoning = item.get("reasoning", None)

    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
                 {"role": "assistant", "content": chosen_reasoning}]
    msg = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    gpt_sft.append({"text": msg})

gpt_sft = Dataset.from_list(gpt_sft)

print("Running SFT on GPT-5.2 Responses …")
sft_args = SFTConfig(
    output_dir=SFT_GPT5_DIR,
    run_name="gpt52_sft",
    learning_rate=5e-6,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=8,
    bf16=True,
    logging_steps=2,
    save_steps=2,
    num_train_epochs=10,
    dataset_text_field="text",
    packing=True,
    eos_token="<|im_end|>"
)
sft_tr = SFTTrainer(model=model, args=sft_args, train_dataset=gpt_sft)
sft_tr.train()

In [ ]:
generate_example_prediction()

### **6. SFT on Silkome**
Now do SFT on the training split of our Silkome dataset, with synthetic hints of silk-relevant motifs.

In [ ]:
# -------------------------------------------
# SFT on Silkome
# -------------------------------------------
print("Loading lamm-mit/silkome-reasoning …")
def extract_json_block(text: str):
    m = re.findall(r"\{[\s\S]*\}", text)
    if not m: return None
    try:
        return json.loads(m[-1])
    except Exception:
        cleaned = re.sub(r"[`’“”]", '"', m[-1]); cleaned = re.sub(r"%", "", cleaned)
        try: return json.loads(cleaned)
        except Exception: return None

def silk_map(ex):
    out = extract_json_block(ex["output"]) or {"strength": None, "toughness": None}
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT_SILK},
            {"role": "user", "content": ex["input"].strip()},
        ],
        "answer": out,
    }


# Read in Silkome data from GitHub
SILKOME_TRAIN_URL = "https://github.com/jaypsiri/Silkome-Reasoning/raw/refs/heads/main/data/silkome_reasoning_dataset/train/data-00000-of-00001.arrow"
SILKOME_TEST_URL= "https://github.com/jaypsiri/Silkome-Reasoning/raw/refs/heads/main/data/silkome_reasoning_dataset/test/data-00000-of-00001.arrow"
silk = load_dataset("arrow", data_files={'train': SILKOME_TRAIN_URL, 'test': SILKOME_TEST_URL})
silkome_train = silk.get("train")
silkome_valid = silk.get("test")


In [ ]:
def aa_stats(seq: str) -> Dict[str, float]:
    s = (seq or "").upper().replace(" ", "")
    L = max(1, len(s))
    def frac(chars): return sum(c in chars for c in s)/L
    # Calculate the fraction of poly-A, GA repeats, GPGGX, and Proline%
    return {
        "poly-A": frac(set("AAA")),
        "GA-repeats":  frac(set("GA")),
        "GPGG":    frac(set("GPGG")),
        "proline":     s.count("P")/L,
        "glycine":     s.count("G")/L,
    }

def synth_reasoning(seq: str, strength: float, toughness: float) -> str:
    # Add synthetic reasoning of relevant motifs to the SFT output column
    st = aa_stats(seq)
    strength_trend = "higher" if strength >= 0.6 else ("lower" if strength <= 0.4 else "moderate")
    toughness_trend = "higher" if toughness >= 0.6 else ("lower" if toughness <= 0.4 else "moderate")

    hints = []
    if st["poly-A"] > 0.08:      hints.append("poly-Alanine contributes to beta-sheet formation and increases strength")
    if st["GA-repeats"] > 0.08:      hints.append("GA-repeats contributes to beta-sheet formation and increases strength")
    if st["proline"] > 0.08:     hints.append("proline can disrupt helices but stabilize beta-turns and increase toughness")
    if st["GPGG"] > 0.08:     hints.append("GPGGX motifs can stabilize beta-turns and increase toughness")
    if st["glycine"] > 0.15:     hints.append("glycine increases flexibility; excessive amounts may reduce stability")
    return (f"Amino-acid composition suggests {strength_trend} strength and {toughness} toughness. "
            f"poly-alanine={st['poly-A']:.2f}, GA-repeats={st['GA-repeats']:.2f}, "
            f"GPGG={st['GPGG']:.2f}, "
            f"P={st['proline']:.2f}, G={st['glycine']:.2f}. " + "; ".join(hints) + ".")

def silkome_sft_row(ex):
    # Format Silkome SFT using the template
    input = ex["input"]
    output = eval(ex["output"])
    seq = re.search(r"Protein sequence: \s*([A-Za-z]+)", input)
    if seq:
      sequence = seq.group(1)
    strength = float(output.get("strength"))
    toughness = float(output.get("toughness"))
    user = input
    assistant_xml = f"<reasoning>\n{synth_reasoning(sequence, strength, toughness)}\n</reasoning>\n<answer>\n{{\"strength\": {strength:.6f}, \"toughness\": {strength:.6f}}}\n</answer>"
    return {"text": apply_chat_template(tokenizer, SYSTEM_PROMPT_SILK, user, assistant_xml)}

silkome_sft = silkome_train.map(silkome_sft_row, remove_columns=[c for c in silkome_train.column_names if c != "text"])

In [ ]:
print("Running SFT on Silkome …")
sft_silkome_args = SFTConfig(
    output_dir=SFT_SILK_DIR,
    run_name="silkome_sft",
    learning_rate=5e-6,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=8,
    bf16=True,
    logging_steps=2,
    save_steps=2,
    num_train_epochs=3,
    dataset_text_field="text",
    packing=True
)
sft_silkome_tr = SFTTrainer(model=model, args=sft_silkome_args, train_dataset=silkome_sft,# tokenizer=tokenizer
                   )
sft_silkome_tr.train()

In [ ]:
generate_example_prediction()

### **7. GRPO on Silkome**

Lastly, do GRPO on our Silkome training dataset. The rewards are the same as previous (format, accuracy, reasoning length) and now we also reward mentions of known important motifs.

In [ ]:
# ============================================================
# GRPO on GB1
# ============================================================
def silk_grpo_row(ex):
    input = ex["input"]
    output = eval(ex["output"])
    seq = re.search(r"Protein sequence: \s*([A-Za-z]+)", input)
    if seq:
      sequence = seq.group(1)
    strength = float(output.get("strength"))
    toughness = float(output.get("toughness"))
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT_SILK},
            {"role": "user", "content": f"Predict normalized strength and toughness in [0,1] with reasoning.\nSequence:\n{sequence}"},
        ],
        "answer": {"strength": strength, "toughness": toughness},
    }
silk_grpo = silkome_train.map(silk_grpo_row, remove_columns=[c for c in silkome_train.column_names if c not in []])

In [ ]:
def json_field_reward_silk(completions, **kwargs):
    outs = parse_json_from_completions(completions)
    scores = []
    for o in outs:
        if not isinstance(o, dict):
            scores.append(0.0); continue
        s = coerce_scalar01(o.get("strength"))
        t = coerce_scalar01(o.get("toughness"))
        scores.append(0.25 if (s is not None and t is not None) else 0.0)
    return scores

def mention_motifs(prompts, completions, answer, **kwargs):
    outs = completions
    scores = []
    for o in outs:
        o_str = str(o)
        temp_score = 0
        if "GPGGX" in o_str or "GPGG" in o_str:
          temp_score += 0.2
        if "GA" in o_str:
          temp_score += 0.1
        if "P" in o_str or "proline" in o_str or "Proline" in o_str:
          temp_score += 0.2
        if "poly-Alanine" in o_str or "poly-alanine" in o_str or "poly-A" in o_str:
          temp_score += 0.2
        scores.append(temp_score)
    return scores

def gated_closeness_silk(prompts, completions, answer, completion_ids=None, **kwargs):
    outs = parse_json_from_completions(completions)
    N = len(outs)
    B = len(answer)
    flat_ids = _flatten_ids(completion_ids)
    scores = []
    for i in range(N):
        t = completions[i][0]["content"]
        r, _ = _extract_sections(t)
        if (r is None) or (len(re.findall(r"\w+|\S", r)) < 60):
            scores.append(0.0); continue
        gold_idx = _gold_index_for_i(i, B, N, flat_ids)
        gold = answer[gold_idx] if 0 <= gold_idx < B else None
        pred = outs[i]
        if not (isinstance(pred, dict) and isinstance(gold, dict)):
            scores.append(0.0); continue
        ps = coerce_scalar01(pred.get("strength"))
        pt = coerce_scalar01(pred.get("toughness"))
        gs = coerce_scalar01(gold.get("strength"))
        gt = coerce_scalar01(gold.get("toughness"))
        if None in (ps, pt, gs, gt):
            scores.append(0.0); continue
        mae = (abs(ps-gs) + abs(pt-gt)) / 2.0
        scores.append(max(0.0, 1.0 - mae))
    return scores

In [ ]:
print("Running GRPO on Silkome …")
grpo_args_silk = GRPOConfig(
    output_dir=GRPO_SILK_DIR,
    run_name="silk_grpo",
    learning_rate=5e-6,
    adam_beta1=0.9, adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    bf16=True,
    per_device_train_batch_size=40,
    gradient_accumulation_steps=8,
    num_generations=8,
    max_completion_length=512,
    num_train_epochs=1,
    save_steps=10,
    max_grad_norm=0.1,
)

silk_tr = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[json_field_reward_silk, anti_shortcut_penalty, gated_closeness_silk, mention_motifs],
    args=grpo_args_silk,
    train_dataset=silk_grpo,
)
silk_tr.train()

In [ ]:
generate_example_prediction()

### **Save Model**

In [ ]:
model.save_pretrained("silkome_reasoning_finetuned")